# Experimental graph network

`roof_from_face_graph` roofs one footprint without a pitch. A network
predicts which faces share a boundary; a planarity step then lifts that
graph. It returns a `Roof` or a `Failure`. Overhang and eave height
keep the meanings they have on `roof`. Gable, knee, gambrel, holes,
dormers, and extra cells are not inputs.

Use it when the wanted roof is a face over several walls, or another
ridge layout. Keep the skeleton when each wall has a pitch.

The shipped checkpoint is loaded; this notebook does not train. After
`uv sync --extra notebooks --extra gnn`.


In [ ]:
from collections import defaultdict

import plotly.graph_objects as go

from krovlab import Failure, Roof, roof
from krovlab.experimental import roof_from_face_graph
from krovlab.viz import plan_view, solid_view


def describe(result: Roof | Failure) -> None:
    if isinstance(result, Failure):
        print(f"Failure  kind={result.kind}\n  {result.reason}")
        return
    by_kind: dict[str, list[float]] = defaultdict(list)
    for arc in result.arcs:
        by_kind[arc.kind].append(arc.length)
    print(f"terrain:           {result.validity.is_terrain}")
    print(f"ridge height:      {result.ridge_height:.3f} m")
    print(f"total sloped area: {result.total_sloped_area:.3f} m²")
    print(f"nodes {len(result.nodes)}, faces {len(result.faces)}, arcs {len(result.arcs)}")
    for kind, lengths in by_kind.items():
        print(f"  {kind:7s}  {len(lengths)} run(s)  total {sum(lengths):.3f} m")
    if not result.validity.is_terrain:
        for reason in result.validity.reasons:
            print(f"  !! {reason}")


def footprint_outline(
    footprint: list[tuple[float, float]],
    title: str = "",
) -> go.Figure:
    """Plan of the input ring — used when there is no roof to draw."""
    fig = go.Figure()
    xs = [p[0] for p in footprint] + [footprint[0][0]]
    ys = [p[1] for p in footprint] + [footprint[0][1]]
    fig.add_trace(
        go.Scatter(
            x=xs, y=ys, mode="lines+markers", name="walls",
            line={"color": "#7c2d12", "width": 2},
            fill="toself", fillcolor="rgba(124, 45, 18, 0.10)",
            marker={"size": 7, "color": "#7c2d12"},
        )
    )
    fig.update_layout(
        title=title, xaxis_title="x (m)", yaxis_title="y (m)",
        yaxis_scaleanchor="x", yaxis_scaleratio=1,
        template="plotly_white", height=420, legend_title="input",
    )
    return fig


def show(
    title: str,
    result: Roof | Failure,
    footprint: list[tuple[float, float]] | None = None,
) -> None:
    """Print quantities, then draw.

    A valid roof: plan + 3D. Brown = walls; grey eave = where the roof
    ends. A Failure: the input outline.
    """
    print(title)
    describe(result)
    if isinstance(result, Roof) and result.validity.is_terrain:
        plan = plan_view(result, walls=footprint)
        plan.update_layout(title=f"{title} — plan (brown = walls, eave = roof edge)")
        plan.show()
        solid = solid_view(result, walls=footprint)
        solid.update_layout(title=f"{title} — 3D (brown = walls at height 0)")
        solid.show()
    elif isinstance(result, Roof):
        plan = plan_view(result, walls=footprint)
        plan.update_layout(title=f"{title} — plan (not a terrain)")
        plan.show()
    elif footprint is not None:
        footprint_outline(footprint, title=f"{title} — {result.kind}").show()


## The same footprint, both ways

An L at 45° through the skeleton, then the same walls through the
experimental method. The checkpoint predicts the face graph. Pitch is
not an input on that second call. Compare quantities and the drawing.


In [ ]:
l_shape = [(0.0, 0.0), (10.0, 0.0), (10.0, 6.0), (3.0, 6.0), (3.0, 10.0), (0.0, 10.0)]

skeleton = roof(l_shape, 45)
experimental = roof_from_face_graph(l_shape)
show("skeleton, 45°", skeleton, l_shape)
show("experimental, checkpoint", experimental, l_shape)


## One plane over several non-collinear walls

Walls 2 and 3 of the L meet at a right angle. The skeleton puts a hip
or a valley there: one face per wall. A supplied face graph can name
those two walls as one face. That face's `node_indices` walk both walls
and the inner corner; the skeleton has six faces, this roof has five.


In [ ]:
mixed = roof_from_face_graph(l_shape, [[0], [1], [2, 3], [4], [5]])
show("one face over walls 2 and 3", mixed, l_shape)
if isinstance(mixed, Roof):
    print(f"faces: {len(mixed.faces)}  (skeleton had 6, one per wall)")
    for face in mixed.faces:
        print(
            f"edge {face.edge_index}: nodes {face.node_indices}  "
            f"plan {face.plan_area:.3f} m²"
        )


## Keep the skeleton when the pitches differ per wall

Pitch is not an input of the experimental method. When each wall has
its own pitch, keep the skeleton: that is the method that takes a
pitch list. The experimental call on the same walls ignores those
angles.


In [ ]:
mixed_pitches = [45, 60, 45, 30, 45, 45]
per_wall = roof(l_shape, mixed_pitches)
show("skeleton, pitches differ per wall", per_wall, l_shape)


## A named Failure

A graph that cannot be lifted is Failure `unliftable`, not a broken
solid. Branch on `kind`; show `reason`. A call with neither a graph
nor a checkpoint is Failure `no_face_graph`.


In [ ]:
missing_walls = roof_from_face_graph(l_shape, [[0], [1]])
show("unliftable: walls missing from the graph", missing_walls, l_shape)
print(missing_walls.kind)
print(missing_walls.reason)
